In [4]:
import json

ENGINEER_MOCK_DATA_LOCATION = 'engineer_mock_data.json' # TODO: user input
TASKS_MOCK_DATA_LOCATION = 'tasks_mock_data.json' # TODO: user input

# read file mock data
def read_file(file_path, validation_func):
    with open(file_path, 'r') as file:
        read_data = json.load(file)
        if (validation_func(read_data)):
            return read_data
        else:
            return []

# validate tasks list and object keys
def validate_tasks(tasks_mock): 
    if not isinstance(tasks_mock, list):
        return False
    for item in tasks_mock:
        if not isinstance(item, dict):
            return False
        if list(item.keys()) != ['task_id', 'skill_required', 'due_date', 'priority', 'dependency_level', 'module_knowledge', 'task_type']:
            return False
    return True

# validate engineer list and object keys
def validate_engineer(engineer_mock):
    if not isinstance(engineer_mock, list):
        return False
    for item in engineer_mock:
        if not isinstance(item, dict):
            return False
        if list(item.keys()) != ['engineer_id', 'skills', 'collab_ability', 'availability_index', 'impact', 'module_exposure', 'efficiency']:
            return False
    return True

engineer_mock = read_file(ENGINEER_MOCK_DATA_LOCATION, validate_engineer)
tasks_mock = read_file(TASKS_MOCK_DATA_LOCATION, validate_tasks)

print(engineer_mock)
print(tasks_mock)

[{'engineer_id': 1, 'skills': 'Spring Boot', 'collab_ability': 'High', 'availability_index': 21, 'impact': 13, 'module_exposure': 5, 'efficiency': 1.23}, {'engineer_id': 2, 'skills': 'React', 'collab_ability': 'Low', 'availability_index': 8, 'impact': 21, 'module_exposure': 13, 'efficiency': 1.79}, {'engineer_id': 3, 'skills': 'React', 'collab_ability': 'Medium', 'availability_index': 13, 'impact': 13, 'module_exposure': 8, 'efficiency': 1.15}, {'engineer_id': 4, 'skills': 'React', 'collab_ability': 'Medium', 'availability_index': 13, 'impact': 21, 'module_exposure': 21, 'efficiency': 1.57}, {'engineer_id': 5, 'skills': 'Angular', 'collab_ability': 'Low', 'availability_index': 8, 'impact': 34, 'module_exposure': 13, 'efficiency': 0.86}, {'engineer_id': 6, 'skills': 'Angular', 'collab_ability': 'High', 'availability_index': 21, 'impact': 13, 'module_exposure': 8, 'efficiency': 1.31}, {'engineer_id': 7, 'skills': 'Spring Boot', 'collab_ability': 'Medium', 'availability_index': 13, 'impac

In [5]:
ENGINEER_SKILL_KEY_NAME = 'skills'
TASK_SKILL_KEY_NAME = 'skill_required'

# sort engineers or tasks by skills 
def sort_items_by_skills(items_list, item_key_name):
    sorted_items_dict = {}
    for item in items_list:
        if not item[item_key_name] in sorted_items_dict:
            sorted_items_dict[item[item_key_name]] = []
        sorted_items_dict[item[item_key_name]].append(item)
    return sorted_items_dict

sorted_engineer_dict = sort_items_by_skills(engineer_mock, ENGINEER_SKILL_KEY_NAME)
sorted_tasks_dict = sort_items_by_skills(tasks_mock, TASK_SKILL_KEY_NAME)

print(sorted_engineer_dict)
print(sorted_tasks_dict)

{'Spring Boot': [{'engineer_id': 1, 'skills': 'Spring Boot', 'collab_ability': 'High', 'availability_index': 21, 'impact': 13, 'module_exposure': 5, 'efficiency': 1.23}, {'engineer_id': 7, 'skills': 'Spring Boot', 'collab_ability': 'Medium', 'availability_index': 13, 'impact': 21, 'module_exposure': 34, 'efficiency': 1.54}, {'engineer_id': 8, 'skills': 'Spring Boot', 'collab_ability': 'Medium', 'availability_index': 13, 'impact': 55, 'module_exposure': 21, 'efficiency': 1.11}, {'engineer_id': 13, 'skills': 'Spring Boot', 'collab_ability': 'Low', 'availability_index': 8, 'impact': 34, 'module_exposure': 5, 'efficiency': 1.12}, {'engineer_id': 15, 'skills': 'Spring Boot', 'collab_ability': 'Low', 'availability_index': 8, 'impact': 13, 'module_exposure': 8, 'efficiency': 1.49}, {'engineer_id': 23, 'skills': 'Spring Boot', 'collab_ability': 'Medium', 'availability_index': 13, 'impact': 13, 'module_exposure': 13, 'efficiency': 1.17}, {'engineer_id': 25, 'skills': 'Spring Boot', 'collab_abil

In [10]:
LEVELS_TO_NUMERICS = {
    'Critical': 4,
    'High': 3,
    'Medium': 2,
    'Low': 1
}
SPRINT_LENGTH = 14 # taking a dummy sprint range of 14 days # TODO: user input
QUARTER_START_DATE = 1711929600 # April 1st 2024 # TODO: user input
QUARTER_END_DATE = 1720396740 # July 7th 2024 (to prevent short sprints) # TODO: user input

def task_complexity_calculation(task):
    return (LEVELS_TO_NUMERICS[task['priority']] + LEVELS_TO_NUMERICS[task['dependency_level']])*task['module_knowledge']

def engineer_complexity_handling_calculation(engineer):
    return LEVELS_TO_NUMERICS[engineer['collab_ability']] * engineer['impact'] * engineer['module_exposure']

def task_time_calculation(task):
    return int(task['due_date']) - QUARTER_START_DATE

def engineer_time_efficiency_calculation(engineer):
    return engineer['availability_index'] * engineer['efficiency'] * 3600

# find time and complexity co-efficients for tasks and engineers
def item_time_complexity_list_generator(item_dict, time_function, complexity_function):
    for skill in list(item_dict.keys()):
        for item in item_dict[skill]:
            item['time_coef'] = time_function(item)
            item['complexity_coef'] = complexity_function(item)
    return item_dict

quantified_engineer_dict = item_time_complexity_list_generator(sorted_engineer_dict, engineer_time_efficiency_calculation, engineer_complexity_handling_calculation)
quantified_task_dict = item_time_complexity_list_generator(sorted_tasks_dict, task_time_calculation, task_complexity_calculation)

print(quantified_engineer_dict)
print(quantified_task_dict)

{'Spring Boot': [{'engineer_id': 1, 'skills': 'Spring Boot', 'collab_ability': 'High', 'availability_index': 21, 'impact': 13, 'module_exposure': 5, 'efficiency': 1.23, 'time_coef': 92988.0, 'complexity_coef': 195}, {'engineer_id': 7, 'skills': 'Spring Boot', 'collab_ability': 'Medium', 'availability_index': 13, 'impact': 21, 'module_exposure': 34, 'efficiency': 1.54, 'time_coef': 72072.0, 'complexity_coef': 1428}, {'engineer_id': 8, 'skills': 'Spring Boot', 'collab_ability': 'Medium', 'availability_index': 13, 'impact': 55, 'module_exposure': 21, 'efficiency': 1.11, 'time_coef': 51948.00000000001, 'complexity_coef': 2310}, {'engineer_id': 13, 'skills': 'Spring Boot', 'collab_ability': 'Low', 'availability_index': 8, 'impact': 34, 'module_exposure': 5, 'efficiency': 1.12, 'time_coef': 32256.000000000004, 'complexity_coef': 170}, {'engineer_id': 15, 'skills': 'Spring Boot', 'collab_ability': 'Low', 'availability_index': 8, 'impact': 13, 'module_exposure': 8, 'efficiency': 1.49, 'time_co

In [ ]:
import matplotlib.pyplot as plt

def plot_engineers_and_tasks_by_skill(engineer_dict, task_dict, skill):
    eng_items = engineer_dict.get(skill, [])
    task_items = task_dict.get(skill, [])

    # Extract coordinates
    eng_x = [e['time_coef'] for e in eng_items]
    eng_y = [e['complexity_coef'] for e in eng_items]

    task_x = [t['time_coef'] for t in task_items]
    task_y = [t['complexity_coef'] for t in task_items]

    plt.figure(figsize=(8, 6))

    # Engineers: Blue
    plt.scatter(eng_x, eng_y, color='red', label='Engineers', marker='^')

    # Tasks: Red
    plt.scatter(task_x, task_y, color='blue', label='Tasks')

    plt.xlabel('Time Coefficient')
    plt.ylabel('Complexity Coefficient')
    plt.title(f"Engineers vs Tasks for Skill: {skill}")
    plt.legend()
    plt.grid(True)
    plt.show()

# plotting the engineers and tasks on same graphs for visualisation
def plot_engineers_and_tasks(quantified_engineer_dict, quantified_task_dict):
    for skill in list(quantified_task_dict.keys()):
        plot_engineers_and_tasks_by_skill(quantified_engineer_dict, quantified_task_dict, skill)

plot_engineers_and_tasks(quantified_engineer_dict, quantified_task_dict)


In [11]:
from datetime import datetime, timedelta

# calculate how many sprints we can have in a given quarter, and the number of days each sprint has
def calculate_sprints(sprint_length_days, quarter_start_ts, quarter_end_ts):
    sprint_length = timedelta(days=sprint_length_days)
    start_date = datetime.fromtimestamp(quarter_start_ts)
    end_date = datetime.fromtimestamp(quarter_end_ts)

    sprints = {}
    current_start = start_date
    sprint_num = 1

    while current_start < end_date:
        current_end = current_start + sprint_length - timedelta(days=1)
        if current_end > end_date:
            current_end = end_date

        sprint_duration = (current_end - current_start).days + 1  # to include both start and end dates

        sprints[sprint_num] = {
            "name": f"Sprint {sprint_num}",
            "start": int(current_start.timestamp()),
            "end": int(current_end.timestamp()),
            "days": sprint_duration
        }

        current_start = current_end + timedelta(days=1)
        sprint_num += 1

    return sprints

sprint_dict = calculate_sprints(SPRINT_LENGTH, QUARTER_START_DATE, QUARTER_END_DATE)

print(sprint_dict)

{1: {'name': 'Sprint 1', 'start': 1711929600, 'end': 1713052800, 'days': 14}, 2: {'name': 'Sprint 2', 'start': 1713139200, 'end': 1714262400, 'days': 14}, 3: {'name': 'Sprint 3', 'start': 1714348800, 'end': 1715472000, 'days': 14}, 4: {'name': 'Sprint 4', 'start': 1715558400, 'end': 1716681600, 'days': 14}, 5: {'name': 'Sprint 5', 'start': 1716768000, 'end': 1717891200, 'days': 14}, 6: {'name': 'Sprint 6', 'start': 1717977600, 'end': 1719100800, 'days': 14}, 7: {'name': 'Sprint 7', 'start': 1719187200, 'end': 1720310400, 'days': 14}}


In [15]:
# calculate task's sprint number
def calculate_task_sprint_number(task, quarter_start, sprint_length):
    needed_days = (int(task['due_date']) - quarter_start) / 86400 # 86400 is the number of seconds in a day
    sprint_number = int((needed_days // sprint_length) + 1) # using floor division to find the full number of sprints, converted to int and +1 to start from 1st sprint
    return sprint_number

# sort tasks to sprints
def sort_tasks_to_sprints(tasks_list, skill, sprint_dict, quarter_start, sprint_length):
    for task in tasks_list:
        sprint_number = calculate_task_sprint_number(task, quarter_start, sprint_length)
        if skill not in sprint_dict[sprint_number]:
            sprint_dict[sprint_number][skill] = {"tasks": [], "engineers": []}
        sprint_dict[sprint_number][skill]["tasks"].append(task)

# sort engineers to sprints
def sort_engineers_to_sprints(engineers_list, skill, sprint_dict):
    for sprint_number in list(sprint_dict.keys()):
        if skill not in sprint_dict[sprint_number]:
            sprint_dict[sprint_number][skill] = {"engineers": [], "tasks": []}
        sprint_dict[sprint_number][skill]["engineers"] = engineers_list

# select tasks and engineers by skills
def select_tasks_engineers_by_skill(quantified_task_dict, quantified_engineer_dict, sprint_dict, quarter_start, sprint_length):
    for skill in list(quantified_task_dict.keys()):
        sort_tasks_to_sprints(quantified_task_dict[skill], skill, sprint_dict, quarter_start, sprint_length)
        sort_engineers_to_sprints(quantified_engineer_dict[skill], skill, sprint_dict)
    return sprint_dict

task_egineer_populated_sprint_list = select_tasks_engineers_by_skill(quantified_task_dict, quantified_engineer_dict, sprint_dict, QUARTER_START_DATE, SPRINT_LENGTH)

print(task_egineer_populated_sprint_list)

{1: {'name': 'Sprint 1', 'start': 1711929600, 'end': 1713052800, 'days': 14, 'Angular': {'tasks': [{'task_id': 14, 'skill_required': 'Angular', 'due_date': '1712087536', 'priority': 'Medium', 'dependency_level': 'High', 'module_knowledge': 8, 'task_type': 'Story', 'time_coef': 157936, 'complexity_coef': 40}, {'task_id': 40, 'skill_required': 'Angular', 'due_date': '1712178369', 'priority': 'Critical', 'dependency_level': 'Low', 'module_knowledge': 21, 'task_type': 'UI Improvement', 'time_coef': 248769, 'complexity_coef': 105}, {'task_id': 65, 'skill_required': 'Angular', 'due_date': '1712787003', 'priority': 'Low', 'dependency_level': 'High', 'module_knowledge': 5, 'task_type': 'UI Improvement', 'time_coef': 857403, 'complexity_coef': 20}, {'task_id': 67, 'skill_required': 'Angular', 'due_date': '1712772869', 'priority': 'Critical', 'dependency_level': 'Low', 'module_knowledge': 21, 'task_type': 'Story', 'time_coef': 843269, 'complexity_coef': 105}, {'task_id': 76, 'skill_required': 'A

In [17]:
import json

with open('task_egineer_populated_sprint_list.json', 'w') as f:
    json.dump(task_egineer_populated_sprint_list, f, indent=4)